# PHASE 4: BỘ QUYẾT ĐỊNH MQTT
## Decision Engine với Calibration

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
import pickle
import time
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.makedirs('Phase4_Models', exist_ok=True)
os.makedirs('Phase4_Data', exist_ok=True)

print("=" * 60)
print("PHASE 4: BỘ QUYẾT ĐỊNH MQTT")
print("=" * 60)

PHASE 4: BỘ QUYẾT ĐỊNH MQTT


## BƯỚC 1: LOAD MODEL

In [2]:
print("\nBƯỚC 1: LOAD MODEL")
print("-" * 40)

with open('Phase2_Models/best_model_info.pkl', 'rb') as f:
    model_info = pickle.load(f)

best_model_name = model_info['best_model_name']
model_filename = model_info['model_filename']

with open(f'Phase2_Models/{model_filename}', 'rb') as f:
    best_model = pickle.load(f)

print(f"Model: {best_model_name}")
print(f"F1-Score: {model_info['test_f1_score']:.4f}")

X_test = pd.read_csv('Phase1_Data/X_test_processed.csv')
y_test = pd.read_csv('Phase1_Data/y_test_processed.csv').iloc[:, 0]


BƯỚC 1: LOAD MODEL
----------------------------------------
Model: LightGBM
F1-Score: 0.9126


## BƯỚC 2: CALIBRATE MODEL

In [3]:
print("\nBƯỚC 2: CALIBRATE MODEL")
print("-" * 40)

X_cal, X_eval, y_cal, y_eval = train_test_split(
    X_test, y_test, test_size=0.8, random_state=42, stratify=y_test
)

calibrated_model = CalibratedClassifierCV(best_model, method='isotonic', cv=3)
calibrated_model.fit(X_cal, y_cal)

print(f"Calibration: {X_cal.shape}, Evaluation: {X_eval.shape}")


BƯỚC 2: CALIBRATE MODEL
----------------------------------------
Calibration: (19858, 33), Evaluation: (79432, 33)


## BƯỚC 3: CLASS MAPPING

In [4]:
print("\nBƯỚC 3: CLASS MAPPING")
print("-" * 40)

with open('Phase1_Models/mqtt_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

CLASS_MAPPING = {i: name for i, name in enumerate(label_encoder.classes_)}

THREAT_LEVELS = {
    'legitimate': 'NORMAL',
    'bruteforce': 'HIGH',
    'dos': 'CRITICAL',
    'flood': 'CRITICAL',
    'malformed': 'MEDIUM',
    'slowite': 'HIGH'
}

print("Classes:", CLASS_MAPPING)


BƯỚC 3: CLASS MAPPING
----------------------------------------
Classes: {0: 'bruteforce', 1: 'dos', 2: 'flood', 3: 'legitimate', 4: 'malformed', 5: 'slowite'}


## BƯỚC 4: DECISION ENGINE

In [5]:
print("\nBƯỚC 4: DECISION ENGINE")
print("-" * 40)

class MQTTDecisionEngine:
    def __init__(self, model, class_mapping, threat_levels, confidence_threshold=0.4):
        self.model = model
        self.class_mapping = class_mapping
        self.threat_levels = threat_levels
        self.confidence_threshold = confidence_threshold
        self.detection_log = []
    
    def predict(self, sample):
        start_time = time.time()
        
        if len(sample.shape) == 1:
            sample = sample.reshape(1, -1)
        elif isinstance(sample, pd.DataFrame):
            sample = sample.values
        
        probabilities = self.model.predict_proba(sample)[0]
        max_prob_idx = np.argmax(probabilities)
        max_confidence = probabilities[max_prob_idx]
        
        if max_confidence < self.confidence_threshold:
            attack_classes = [i for i, name in self.class_mapping.items() if name != 'legitimate']
            if attack_classes:
                attack_probs = [(i, probabilities[i]) for i in attack_classes]
                attack_probs.sort(key=lambda x: x[1], reverse=True)
                if attack_probs[0][1] > 0.25:
                    predicted_class = attack_probs[0][0]
                    confidence = attack_probs[0][1]
                else:
                    predicted_class = max_prob_idx
                    confidence = max_confidence
            else:
                predicted_class = max_prob_idx
                confidence = max_confidence
        else:
            predicted_class = max_prob_idx
            confidence = max_confidence
        
        attack_name = self.class_mapping.get(predicted_class, 'unknown')
        threat_level = self.threat_levels.get(attack_name, 'UNKNOWN')
        processing_time = (time.time() - start_time) * 1000
        
        result = {
            'prediction': predicted_class,
            'attack_name': attack_name,
            'confidence': confidence,
            'threat_level': threat_level,
            'processing_time_ms': processing_time
        }
        
        self.detection_log.append(result)
        return result

mqtt_engine = MQTTDecisionEngine(
    model=calibrated_model,
    class_mapping=CLASS_MAPPING,
    threat_levels=THREAT_LEVELS
)

print("Engine created")


BƯỚC 4: DECISION ENGINE
----------------------------------------
Engine created


## BƯỚC 5: ĐÁNH GIÁ

In [6]:
print("\nBƯỚC 5: ĐÁNH GIÁ")
print("-" * 40)

# Use batch prediction instead of loop (much faster)
all_predictions = mqtt_engine.model.predict(X_eval)

accuracy = accuracy_score(y_eval, all_predictions)
precision = precision_score(y_eval, all_predictions, average='weighted', zero_division=0)
recall = recall_score(y_eval, all_predictions, average='weighted', zero_division=0)
f1 = f1_score(y_eval, all_predictions, average='weighted', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


BƯỚC 5: ĐÁNH GIÁ
----------------------------------------
Accuracy: 0.9312
Precision: 0.9329
Recall: 0.9312
F1-Score: 0.9306


## BƯỚC 6: LƯU KẾT QUẢ

In [7]:
print("\nBƯỚC 6: LƯU KẾT QUẢ")
print("-" * 40)

with open('Phase4_Models/mqtt_decision_engine.pkl', 'wb') as f:
    pickle.dump(mqtt_engine, f)

final_metrics = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'model_type': best_model_name
}

with open('Phase4_Data/final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print("✅ Đã lưu engine và metrics")


BƯỚC 6: LƯU KẾT QUẢ
----------------------------------------
✅ Đã lưu engine và metrics


## BƯỚC 7: TEST TRÊN RAW DATASET

In [8]:
print("\nBƯỚC 7: TEST TRÊN RAW DATASET")
print("-" * 40)

# Load raw test data
test_raw = pd.read_csv('MQTT/test30_reduced.csv')
print(f"Raw test data: {test_raw.shape}")

# Load preprocessing artifacts
with open('Phase1_Models/mqtt_feature_encoders.pkl', 'rb') as f:
    feature_encoders = pickle.load(f)
with open('Phase1_Models/mqtt_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
with open('Phase1_Models/mqtt_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Separate features and target
X_test_raw = test_raw.drop('target', axis=1)
y_test_raw = test_raw['target']

# Apply preprocessing
X_test_encoded = X_test_raw.copy()

# Encode categorical features
for col, encoder in feature_encoders.items():
    if col in X_test_encoded.columns:
        mapping = {label: idx for idx, label in enumerate(encoder.classes_)}
        X_test_encoded[col] = X_test_encoded[col].map(lambda x: mapping.get(x, -1))

# Scale features
X_test_scaled = scaler.transform(X_test_encoded)

# Encode labels
y_test_encoded = label_encoder.transform(y_test_raw)

# Predict with decision engine (batch prediction for speed)
all_predictions_raw = mqtt_engine.model.predict(X_test_scaled)

# Evaluate
accuracy_raw = accuracy_score(y_test_encoded, all_predictions_raw)
precision_raw = precision_score(y_test_encoded, all_predictions_raw, average='weighted', zero_division=0)
recall_raw = recall_score(y_test_encoded, all_predictions_raw, average='weighted', zero_division=0)
f1_raw = f1_score(y_test_encoded, all_predictions_raw, average='weighted', zero_division=0)

print(f"\nKết quả test trên RAW dataset:")
print(f"Accuracy: {accuracy_raw:.4f}")
print(f"Precision: {precision_raw:.4f}")
print(f"Recall: {recall_raw:.4f}")
print(f"F1-Score: {f1_raw:.4f}")

print(f"\nSo sánh Processed vs Raw:")
print(f"  Processed F1: {f1:.4f}")
print(f"  Raw F1: {f1_raw:.4f}")
print(f"  Difference: {abs(f1 - f1_raw):.4f}")


BƯỚC 7: TEST TRÊN RAW DATASET
----------------------------------------
Raw test data: (99290, 34)

Kết quả test trên RAW dataset:
Accuracy: 0.9327
Precision: 0.9345
Recall: 0.9327
F1-Score: 0.9320

So sánh Processed vs Raw:
  Processed F1: 0.9306
  Raw F1: 0.9320
  Difference: 0.0014


In [9]:
print("\n" + "=" * 60)
print("HOÀN THÀNH PHASE 4!")
print("=" * 60)
print(f"F1 (Processed): {f1:.4f}")
print(f"F1 (Raw): {f1_raw:.4f}")


HOÀN THÀNH PHASE 4!
F1 (Processed): 0.9306
F1 (Raw): 0.9320
